# 从 PokemonDB 抓取第九世代招式数据

本 notebook 会从 https://pokemondb.net/move/generation/9 抓取第九世代全部招式的基础信息，并导出为 `../data/move_gen9.csv`。

In [ ]:
import json
import re
import time
from pathlib import Path

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

BASE_URL = 'https://pokemondb.net'
MOVES_URL = f'{BASE_URL}/move/generation/9'
OUTPUT_JSON = Path('../data/move_gen9.json').resolve()
OUTPUT_CSV = Path('../data/move_gen9.csv').resolve()
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

session = requests.Session()
session.headers.update(HEADERS)

def get_soup(url: str):
    response = session.get(url, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')

def parse_move_list():
    soup = get_soup(MOVES_URL)
    rows = soup.select('table.data-table tbody tr')
    moves = []
    for index, row in enumerate(rows, start=1):
        cells = row.select('td')
        if len(cells) < 7:
            continue
        name_link = row.select_one('a.ent-name')
        if not name_link:
            continue
        name_en = name_link.get_text(strip=True)
        href = name_link.get('href', '')
        slug = href.strip('/').split('/')[-1] if href else ''
        detail_url = f'{BASE_URL}{href}' if href else ''
        move_type = cells[1].get_text(strip=True)
        category = cells[2].get_text(strip=True)
        power = cells[3].get_text(strip=True)
        accuracy = cells[4].get_text(strip=True)
        pp = cells[5].get_text(strip=True)
        effect = cells[6].get_text(strip=True)
        moves.append({
            'move_id': index,
            'name_en': name_en,
            'slug': slug,
            'detail_url': detail_url,
            'type': move_type,
            'category': category,
            'power': power,
            'accuracy': accuracy,
            'pp': pp,
            'effect': effect,
            'name_cn': '',
        })
    return moves

def parse_chinese_name(soup):
    heading = soup.find(lambda tag: tag.name in ['h2', 'h3'] and 'Other languages' in tag.get_text())
    if heading is None:
        return ''
    table = heading.find_next('table')
    if table is None:
        return ''
    for row in table.select('tr'):
        th = row.find('th')
        td = row.find('td')
        if not th or not td:
            continue
        if th.get_text(strip=True) == 'Chinese (Simplified)':
            return td.get_text(strip=True)
    return ''

def parse_effect_text(soup):
    heading = soup.find(lambda tag: tag.name in ['h2', 'h3'] and 'Effects' in tag.get_text())
    if heading is None:
        return ''
    texts = []
    sibling = heading.find_next_sibling()
    while sibling is not None and sibling.name not in ['h2', 'h3']:
        if sibling.name in ['p', 'div', 'ul', 'ol']:
            text = sibling.get_text(separator=' ', strip=True)
            if text:
                texts.append(text)
        sibling = sibling.find_next_sibling()
    return ' '.join(texts).strip()

def parse_move_detail(move):
    if not move['detail_url']:
        return move
    soup = get_soup(move['detail_url'])
    name_cn = parse_chinese_name(soup)
    effect_full = parse_effect_text(soup)
    move['name_cn'] = name_cn
    move['effect'] = effect_full or move['effect']
    return move

def fetch_all_moves():
    moves = parse_move_list()
    results = []
    for index, move in enumerate(moves, start=1):
        print(f"Fetching {index}/{len(moves)}: {move['name_en']}")
        detail_move = parse_move_detail(move)
        results.append(detail_move)
        time.sleep(0.5)
    return results

def save_csv(move_data):
    import csv
    fieldnames = [
        'move_id',
        'name_en',
        'name_cn',
        'type',
        'category',
        'power',
        'accuracy',
        'pp',
        'effect',
    ]
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_CSV, 'w', encoding='utf-8', newline='') as fp:
        writer = csv.DictWriter(fp, fieldnames=fieldnames)
        writer.writeheader()
        for row in move_data:
            writer.writerow({key: row.get(key, '') for key in fieldnames})

move_data = fetch_all_moves()
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_JSON, 'w', encoding='utf-8') as fp:
    json.dump(move_data, fp, ensure_ascii=False, indent=2)
save_csv(move_data)
print(f'完成：已保存 {len(move_data)} 条招式数据到 {OUTPUT_JSON} 和 {OUTPUT_CSV}')

NameError: name 'name_en' is not defined